# (3) Data Generation and Processing

This chapter details the comprehensive data generation pipeline for training RNN-based performance map prediction models. We focus on efficient design space exploration through Latin Hypercube Sampling (LHS), the 4-stage finite element (FE) simulation process, and the transformation of simulation results into sequence data suitable for RNN training.

## Learning Objectives

- Understand Latin Hypercube Sampling for efficient design space exploration
- Master the 4-stage FE simulation workflow for motor performance analysis
- Learn sequence data representation techniques for RNN training
- Implement data preprocessing and normalization strategies
- Explore data augmentation and synthetic data generation techniques

## 3.1 Design Space Exploration with Latin Hypercube Sampling

### 3.1.1 Challenge of High-Dimensional Design Spaces

Electric motor design involves multiple geometric and control parameters that span a high-dimensional design space. Traditional grid-based sampling becomes computationally infeasible as the number of parameters increases.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import seaborn as sns
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

def compare_sampling_methods(n_samples=100, n_params=2, seed=42):
    """
    Compare different sampling methods for design space exploration
    """
    np.random.seed(seed)
    
    # Latin Hypercube Sampling
    lhs_sampler = qmc.LatinHypercube(d=n_params, seed=seed)
    lhs_samples = lhs_sampler.random(n=n_samples)
    
    # Random Sampling
    random_samples = np.random.random((n_samples, n_params))
    
    # Grid Sampling (for comparison, fewer samples due to curse of dimensionality)
    grid_points_per_dim = int(np.sqrt(n_samples))
    grid_range = np.linspace(0.1, 0.9, grid_points_per_dim)
    grid_samples = np.array(list(product(grid_range, repeat=n_params)))
    
    return lhs_samples, random_samples, grid_samples

def plot_sampling_comparison(lhs_samples, random_samples, grid_samples):
    """
    Visualize comparison of different sampling methods
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # Latin Hypercube Sampling
    axes[0, 0].scatter(lhs_samples[:, 0], lhs_samples[:, 1], 
                     alpha=0.6, s=50, c='blue', edgecolors='black', linewidth=0.5)
    axes[0, 0].set_title('Latin Hypercube Sampling', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Parameter 1')
    axes[0, 0].set_ylabel('Parameter 2')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Random Sampling
    axes[0, 1].scatter(random_samples[:, 0], random_samples[:, 1], 
                     alpha=0.6, s=50, c='red', edgecolors='black', linewidth=0.5)
    axes[0, 1].set_title('Random Sampling', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Parameter 1')
    axes[0, 1].set_ylabel('Parameter 2')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Grid Sampling
    axes[1, 0].scatter(grid_samples[:, 0], grid_samples[:, 1], 
                     alpha=0.6, s=50, c='green', edgecolors='black', linewidth=0.5)
    axes[1, 0].set_title('Grid Sampling', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Parameter 1')
    axes[1, 0].set_ylabel('Parameter 2')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Coverage Analysis
    def calculate_coverage(samples, n_bins=10):
        """Calculate percentage of occupied bins"""
        hist, _, _ = np.histogram2d(samples[:, 0], samples[:, 1], bins=n_bins, 
                                  range=[[0, 1], [0, 1]])
        occupied_bins = np.sum(hist > 0)
        total_bins = n_bins * n_bins
        return (occupied_bins / total_bins) * 100
    
    methods = ['Latin Hypercube', 'Random', 'Grid']
    coverages = [
        calculate_coverage(lhs_samples),
        calculate_coverage(random_samples),
        calculate_coverage(grid_samples)
    ]
    
    bars = axes[1, 1].bar(methods, coverages, color=['blue', 'red', 'green'], alpha=0.7)
    axes[1, 1].set_title('Design Space Coverage', fontsize=14, fontweight='bold')
    axes[1, 1].set_ylabel('Coverage (%)')
    axes[1, 1].set_ylim(0, 100)
    
    # Add value labels on bars
    for bar, coverage in zip(bars, coverages):
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height + 1,
                       f'{coverage:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Compare sampling methods
lhs_samples, random_samples, grid_samples = compare_sampling_methods(n_samples=100, n_params=2)
plot_sampling_comparison(lhs_samples, random_samples, grid_samples)

### 3.1.2 Latin Hypercube Sampling Implementation

LHS provides excellent space-filling properties with relatively few samples, making it ideal for expensive FE simulations.

In [ ]:
class MotorDesignSampler:
    """
    Latin Hypercube Sampling for motor design parameters
    """
    
    def __init__(self, seed=42):
        self.seed = seed
        
        # Define motor design parameter ranges
        self.parameter_ranges = {
            # Geometric parameters
            'stator_outer_diameter': (100, 200),  # mm
            'stator_inner_diameter': (60, 120),   # mm
            'rotor_outer_diameter': (50, 110),   # mm
            'air_gap_length': (0.5, 2.0),        # mm
            'stack_length': (50, 150),           # mm
            'slot_depth': (8, 15),               # mm
            'slot_opening': (2, 5),              # mm
            'tooth_width': (3, 8),               # mm
            'back_iron_thickness': (5, 12),      # mm
            
            # Winding parameters
            'turns_per_coil': (5, 20),           # turns
            'wire_diameter': (0.5, 2.0),         # mm
            'parallel_paths': (1, 4),            # integer
            
            # Material parameters
            'steel_grade': (1, 5),               # discrete grades
            'magnet_grade': (1, 3),              # discrete grades
            'magnet_thickness': (2, 8),          # mm
            
            # Control parameters
            'base_speed': (1000, 5000),          # RPM
            'max_current': (50, 200),            # A
            'max_voltage': (200, 650),           # V
            'dc_bus_voltage': (200, 800)         # V
        }
        
        self.parameter_names = list(self.parameter_ranges.keys())
        self.n_params = len(self.parameter_names)
    
    def generate_samples(self, n_samples):
        """
        Generate Latin Hypercube samples for motor design parameters
        """
        # Create LHS sampler
        sampler = qmc.LatinHypercube(d=self.n_params, seed=self.seed)
        
        # Generate samples in unit hypercube
        lhs_samples = sampler.random(n=n_samples)
        
        # Scale to parameter ranges
        scaled_samples = np.zeros_like(lhs_samples)
        
        for i, param_name in enumerate(self.parameter_names):
            min_val, max_val = self.parameter_ranges[param_name]
            scaled_samples[:, i] = lhs_samples[:, i] * (max_val - min_val) + min_val
            
            # Handle integer parameters
            if param_name in ['parallel_paths', 'steel_grade', 'magnet_grade']:
                scaled_samples[:, i] = np.round(scaled_samples[:, i]).astype(int)
        
        # Create DataFrame with parameter names
        samples_df = pd.DataFrame(scaled_samples, columns=self.parameter_names)
        
        return samples_df, lhs_samples
    
    def analyze_sample_quality(self, lhs_samples):
        """
        Analyze the quality of LHS samples
        """
        # Calculate pairwise correlations
        correlations = np.corrcoef(lhs_samples.T)
        
        # Calculate minimum distance between samples
        from scipy.spatial.distance import pdist
        min_distances = pdist(lhs_samples)
        min_distance = np.min(min_distances)
        mean_distance = np.mean(min_distances)
        
        # Calculate maximum correlation (off-diagonal)
        corr_matrix = np.abs(correlations)
        np.fill_diagonal(corr_matrix, 0)
        max_correlation = np.max(corr_matrix)
        
        quality_metrics = {
            'min_distance': min_distance,
            'mean_distance': mean_distance,
            'max_correlation': max_correlation,
            'space_filling': mean_distance / min_distance if min_distance > 0 else 0
        }
        
        return quality_metrics, correlations
    
    def visualize_parameter_distributions(self, samples_df):
        """
        Visualize parameter distributions
        """
        n_params = len(self.parameter_names)
        n_cols = 4
        n_rows = (n_params + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
        axes = axes.flatten() if n_rows > 1 else [axes] if n_cols > 1 else [axes]
        
        for i, param_name in enumerate(self.parameter_names):
            ax = axes[i]
            
            # Plot histogram
            ax.hist(samples_df[param_name], bins=20, alpha=0.7, edgecolor='black')
            ax.set_title(f'{param_name.replace("_", " ").title()}', fontsize=10)
            ax.set_xlabel('Value')
            ax.set_ylabel('Frequency')
            ax.grid(True, alpha=0.3)
        
        # Hide unused subplots
        for i in range(n_params, len(axes)):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        plt.show()

# Generate motor design samples
sampler = MotorDesignSampler(seed=42)
samples_df, lhs_samples = sampler.generate_samples(n_samples=500)

# Analyze sample quality
quality_metrics, correlations = sampler.analyze_sample_quality(lhs_samples)

print("Sample Quality Metrics:")
for metric, value in quality_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Visualize parameter distributions
sampler.visualize_parameter_distributions(samples_df)

## 3.2 4-Stage Finite Element Simulation Process

The 4-stage FE simulation process generates comprehensive performance data for each motor design. Each stage focuses on different aspects of motor performance and operates at different computational costs.

In [ ]:
class FESimulationStages:
    """
    4-Stage FE Simulation Process for Motor Performance Analysis
    """
    
    def __init__(self):
        self.stage_info = {
            'magnetic_analysis': {
                'description': 'Magnetic field distribution and flux linkage',
                'computational_cost': 'High',
                'outputs': ['flux_density', 'flux_linkage', 'inductance'],
                'time_per_point': '5-10 minutes'
            },
            'thermal_analysis': {
                'description': 'Temperature distribution and thermal behavior',
                'computational_cost': 'Medium',
                'outputs': ['temperature', 'thermal_resistance', 'heat_flow'],
                'time_per_point': '2-5 minutes'
            },
            'mechanical_analysis': {
                'description': 'Mechanical stress and vibration analysis',
                'computational_cost': 'Low',
                'outputs': ['stress', 'deformation', 'natural_frequency'],
                'time_per_point': '30-60 seconds'
            },
            'performance_analysis': {
                'description': 'Efficiency, torque, and power characteristics',
                'computational_cost': 'Low',
                'outputs': ['efficiency', 'torque', 'power_factor', 'losses'],
                'time_per_point': '10-30 seconds'
            }
        }
    
    def simulate_stage(self, stage_name, design_params, operating_point):
        """
        Simulate a specific FE stage (simplified for demonstration)
        """
        np.random.seed(hash(str(design_params) + str(operating_point)) % 2**32)
        
        if stage_name == 'magnetic_analysis':
            return self._magnetic_simulation(design_params, operating_point)
        elif stage_name == 'thermal_analysis':
            return self._thermal_simulation(design_params, operating_point)
        elif stage_name == 'mechanical_analysis':
            return self._mechanical_simulation(design_params, operating_point)
        elif stage_name == 'performance_analysis':
            return self._performance_simulation(design_params, operating_point)
    
    def _magnetic_simulation(self, design_params, operating_point):
        """
        Simplified magnetic simulation
        """
        speed, current = operating_point
        
        # Base flux linkage calculation
        base_flux = 0.1 * design_params['stack_length'] * design_params['turns_per_coil']
        flux_density = base_flux * (1 + 0.1 * np.sin(speed * current / 1000))
        
        # Inductance calculation
        l_d = 0.001 * design_params['turns_per_coil']**2 * design_params['stack_length'] / design_params['air_gap_length']
        l_q = 0.0008 * design_params['turns_per_coil']**2 * design_params['stack_length'] / design_params['air_gap_length']
        
        return {
            'flux_density': flux_density,
            'flux_linkage_d': l_d * current,
            'flux_linkage_q': l_q * current,
            'inductance_d': l_d,
            'inductance_q': l_q
        }
    
    def _thermal_simulation(self, design_params, operating_point):
        """
        Simplified thermal simulation
        """
        speed, current = operating_point
        
        # Copper losses
        resistance = 0.01 * design_params['turns_per_coil'] * design_params['wire_diameter']**-2
        copper_losses = 3 * current**2 * resistance
        
        # Core losses (simplified Steinmetz equation)
        flux_density = 1.2  # Tesla (typical value)
        frequency = speed / 60
        core_losses = 0.005 * flux_density**2 * frequency**1.5 * design_params['stack_length']
        
        # Temperature rise
        total_losses = copper_losses + core_losses
        thermal_resistance = 0.5 / design_params['stack_length']  # K/W
        temperature_rise = total_losses * thermal_resistance
        
        return {
            'copper_losses': copper_losses,
            'core_losses': core_losses,
            'total_losses': total_losses,
            'temperature_rise': temperature_rise,
            'thermal_resistance': thermal_resistance
        }
    
    def _mechanical_simulation(self, design_params, operating_point):
        """
        Simplified mechanical simulation
        """
        speed, current = operating_point
        
        # Rotor stress (centrifugal force)
        rotor_radius = design_params['rotor_outer_diameter'] / 2000  # Convert to meters
        density = 7850  # kg/m³ (steel)
        stress = density * (2 * np.pi * speed / 60)**2 * rotor_radius**2 / 3
        
        # Natural frequency (simplified)
        young_modulus = 200e9  # Pa (steel)
        moment_of_inertia = np.pi * rotor_radius**4 / 4
        natural_freq = np.sqrt(young_modulus * moment_of_inertia / density) / (2 * np.pi * rotor_radius**2)
        
        return {
            'rotor_stress': stress,
            'natural_frequency': natural_freq,
            'safety_factor': 200e6 / stress if stress > 0 else float('inf')  # 200 MPa yield strength
        }
    
    def _performance_simulation(self, design_params, operating_point):
        """
        Simplified performance simulation
        """
        speed, current = operating_point
        
        # Torque calculation (simplified)
        torque_constant = 0.05 * design_params['stack_length'] * design_params['turns_per_coil']
        torque = torque_constant * current * (1 - 0.001 * speed / 1000)  # Speed derating
        
        # Power calculation
        mechanical_power = torque * 2 * np.pi * speed / 60
        
        # Efficiency calculation
        resistance = 0.01 * design_params['turns_per_coil'] * design_params['wire_diameter']**-2
        copper_losses = 3 * current**2 * resistance
        
        flux_density = 1.2
        frequency = speed / 60
        core_losses = 0.005 * flux_density**2 * frequency**1.5 * design_params['stack_length']
        
        total_losses = copper_losses + core_losses
        electrical_power = mechanical_power + total_losses
        efficiency = mechanical_power / electrical_power if electrical_power > 0 else 0
        
        # Power factor
        reactance = 2 * np.pi * frequency * 0.001 * design_params['turns_per_coil']**2
        impedance = np.sqrt(resistance**2 + reactance**2)
        power_factor = resistance / impedance
        
        return {
            'torque': torque,
            'mechanical_power': mechanical_power,
            'electrical_power': electrical_power,
            'efficiency': efficiency,
            'power_factor': power_factor,
            'copper_losses': copper_losses,
            'core_losses': core_losses
        }
    
    def run_full_simulation(self, design_params, speed_points, current_points):
        """
        Run complete 4-stage simulation for multiple operating points
        """
        results = {
            'design_params': design_params,
            'operating_points': [],
            'magnetic_results': [],
            'thermal_results': [],
            'mechanical_results': [],
            'performance_results': []
        }
        
        for speed in speed_points:
            for current in current_points:
                operating_point = (speed, current)
                
                # Run all 4 stages
                magnetic_results = self.simulate_stage('magnetic_analysis', design_params, operating_point)
                thermal_results = self.simulate_stage('thermal_analysis', design_params, operating_point)
                mechanical_results = self.simulate_stage('mechanical_analysis', design_params, operating_point)
                performance_results = self.simulate_stage('performance_analysis', design_params, operating_point)
                
                results['operating_points'].append(operating_point)
                results['magnetic_results'].append(magnetic_results)
                results['thermal_results'].append(thermal_results)
                results['mechanical_results'].append(mechanical_results)
                results['performance_results'].append(performance_results)
        
        return results

# Initialize FE simulation
fe_sim = FESimulationStages()

# Display stage information
print("4-Stage FE Simulation Process:")
print("=" * 50)
for stage, info in fe_sim.stage_info.items():
    print(f"\n{stage.replace('_', ' ').title()}:")
    print(f"  Description: {info['description']}")
    print(f"  Computational Cost: {info['computational_cost']}")
    print(f"  Time per Point: {info['time_per_point']}")
    print(f"  Outputs: {', '.join(info['outputs'])}")

### 3.2.1 Demonstration of 4-Stage Simulation Process

Let's demonstrate the complete 4-stage simulation process for a sample motor design.

In [ ]:
# Sample motor design parameters
sample_design = {
    'stator_outer_diameter': 150,
    'stator_inner_diameter': 90,
    'rotor_outer_diameter': 85,
    'air_gap_length': 1.0,
    'stack_length': 100,
    'slot_depth': 10,
    'slot_opening': 3,
    'tooth_width': 5,
    'back_iron_thickness': 8,
    'turns_per_coil': 12,
    'wire_diameter': 1.2,
    'parallel_paths': 2,
    'steel_grade': 2,
    'magnet_grade': 2,
    'magnet_thickness': 4,
    'base_speed': 3000,
    'max_current': 120,
    'max_voltage': 400,
    'dc_bus_voltage': 650
}

# Define operating points for simulation
speed_points = np.linspace(1000, 6000, 10)  # RPM
current_points = np.linspace(20, 150, 8)    # A

print(f"Running 4-stage FE simulation for {len(speed_points) * len(current_points)} operating points...")
print(f"Speed range: {speed_points[0]:.0f} - {speed_points[-1]:.0f} RPM")
print(f"Current range: {current_points[0]:.0f} - {current_points[-1]:.0f} A")

# Run full simulation
simulation_results = fe_sim.run_full_simulation(sample_design, speed_points, current_points)

print(f"\nSimulation completed!")
print(f"Total data points generated: {len(simulation_results['operating_points'])}")

# Display sample results
sample_idx = 40  # Sample operating point
sample_op = simulation_results['operating_points'][sample_idx]
sample_perf = simulation_results['performance_results'][sample_idx]

print(f"\nSample Results at {sample_op[0]:.0f} RPM, {sample_op[1]:.0f} A:")
print(f"  Torque: {sample_perf['torque']:.2f} Nm")
print(f"  Power: {sample_perf['mechanical_power']/1000:.2f} kW")
print(f"  Efficiency: {sample_perf['efficiency']*100:.1f}%")
print(f"  Power Factor: {sample_perf['power_factor']:.3f}")

### 3.2.2 Computational Cost Analysis

The 4-stage process has different computational costs for each stage. Let's analyze the time requirements and optimize the simulation strategy.

In [ ]:
def analyze_computational_cost(n_designs, n_speed_points, n_current_points):
    """
    Analyze computational cost of 4-stage FE simulation
    """
    # Time per operating point (in minutes)
    stage_times = {
        'magnetic_analysis': 7.5,
        'thermal_analysis': 3.5,
        'mechanical_analysis': 0.75,
        'performance_analysis': 0.25
    }
    
    total_op_points = n_designs * n_speed_points * n_current_points
    
    cost_analysis = {
        'total_designs': n_designs,
        'total_operating_points': total_op_points,
        'stage_costs': {},
        'total_time_hours': 0,
        'optimization_strategies': []
    }
    
    # Calculate costs per stage
    for stage, time_per_point in stage_times.items():
        stage_time = total_op_points * time_per_point
        cost_analysis['stage_costs'][stage] = {
            'time_minutes': stage_time,
            'time_hours': stage_time / 60,
            'percentage': 100 * stage_time / sum(stage_times.values())
        }
        cost_analysis['total_time_hours'] += stage_time / 60
    
    # Optimization strategies
    if cost_analysis['total_time_hours'] > 100:
        cost_analysis['optimization_strategies'].append('Use parallel computing')
    if cost_analysis['total_time_hours'] > 1000:
        cost_analysis['optimization_strategies'].append('Implement adaptive sampling')
    if n_speed_points > 20:
        cost_analysis['optimization_strategies'].append('Reduce speed point density')
    if n_current_points > 15:
        cost_analysis['optimization_strategies'].append('Use logarithmic current spacing')
    
    return cost_analysis

def plot_computational_cost(cost_analysis):
    """
    Visualize computational cost breakdown
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Cost breakdown by stage
    stages = list(cost_analysis['stage_costs'].keys())
    times = [cost_analysis['stage_costs'][stage]['time_hours'] for stage in stages]
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
    
    bars = ax1.bar(stages, times, color=colors, alpha=0.8, edgecolor='black')
    ax1.set_title('Computational Cost by Stage', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Time (hours)')
    ax1.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar, time in zip(bars, times):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + max(times)*0.01,
                f'{time:.1f}h', ha='center', va='bottom', fontweight='bold')
    
    # Time requirements for different scenarios
    scenarios = [
        (10, 10, 8),   # Small dataset
        (50, 20, 12),   # Medium dataset
        (100, 30, 15),  # Large dataset
        (500, 40, 20),  # Very large dataset
    ]
    
    scenario_names = ['Small', 'Medium', 'Large', 'Very Large']
    scenario_times = []
    
    for n_designs, n_speed, n_current in scenarios:
        analysis = analyze_computational_cost(n_designs, n_speed, n_current)
        scenario_times.append(analysis['total_time_hours'])
    
    bars2 = ax2.bar(scenario_names, scenario_times, color='lightcoral', 
                    alpha=0.8, edgecolor='black')
    ax2.set_title('Total Time for Different Dataset Sizes', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Total Time (hours)')
    ax2.set_yscale('log')
    
    # Add value labels
    for bar, time in zip(bars2, scenario_times):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height * 1.1,
                f'{time:.1f}h', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Analyze computational cost for our dataset
n_designs = 100
n_speed_points = 20
n_current_points = 12

cost_analysis = analyze_computational_cost(n_designs, n_speed_points, n_current_points)

print("Computational Cost Analysis:")
print(f"=" * 40)
print(f"Number of designs: {n_designs}")
print(f"Speed points: {n_speed_points}")
print(f"Current points: {n_current_points}")
print(f"Total operating points: {cost_analysis['total_operating_points']:,}")
print(f"\nStage-by-stage breakdown:")

for stage, cost in cost_analysis['stage_costs'].items():
    print(f"  {stage.replace('_', ' ').title()}: {cost['time_hours']:.1f} hours ({cost['percentage']:.1f}%)")

print(f"\nTotal simulation time: {cost_analysis['total_time_hours']:.1f} hours ({cost_analysis['total_time_hours']/24:.1f} days)")

if cost_analysis['optimization_strategies']:
    print("\nRecommended optimization strategies:")
    for strategy in cost_analysis['optimization_strategies']:
        print(f"  - {strategy}")

plot_computational_cost(cost_analysis)

## 3.3 Sequence Data Representation for RNN Training

RNN models require sequence data rather than independent data points. We need to transform the FE simulation results into sequences that capture the relationships between operating points and design parameters.

In [ ]:
class SequenceDataGenerator:
    """
    Convert FE simulation results to sequence data for RNN training
    """
    
    def __init__(self):
        self.sequence_types = {
            'speed_sequence': 'Fixed current, varying speed',
            'current_sequence': 'Fixed speed, varying current',
            'power_sequence': 'Fixed power ratio, varying operating point',
            'efficiency_sequence': 'Contour-based efficiency tracking',
            'mixed_sequence': 'Random operating point transitions'
        }
    
    def create_speed_sequences(self, simulation_results, current_values):
        """
        Create sequences with fixed current, varying speed
        """
        sequences = []
        
        for current in current_values:
            # Find operating points with this current
            current_indices = [i for i, (speed, curr) in 
                             enumerate(simulation_results['operating_points']) 
                             if abs(curr - current) < 1e-6]
            
            if len(current_indices) > 1:
                # Sort by speed
                current_indices.sort(key=lambda i: simulation_results['operating_points'][i][0])
                
                # Extract sequence data
                sequence = {
                    'type': 'speed_sequence',
                    'fixed_parameter': ('current', current),
                    'speeds': [simulation_results['operating_points'][i][0] for i in current_indices],
                    'torque': [simulation_results['performance_results'][i]['torque'] for i in current_indices],
                    'efficiency': [simulation_results['performance_results'][i]['efficiency'] for i in current_indices],
                    'power_factor': [simulation_results['performance_results'][i]['power_factor'] for i in current_indices],
                    'losses': [simulation_results['thermal_results'][i]['total_losses'] for i in current_indices],
                    'flux_density': [simulation_results['magnetic_results'][i]['flux_density'] for i in current_indices]
                }
                sequences.append(sequence)
        
        return sequences
    
    def create_current_sequences(self, simulation_results, speed_values):
        """
        Create sequences with fixed speed, varying current
        """
        sequences = []
        
        for speed in speed_values:
            # Find operating points with this speed
            speed_indices = [i for i, (s, current) in 
                           enumerate(simulation_results['operating_points']) 
                           if abs(s - speed) < 1e-6]
            
            if len(speed_indices) > 1:
                # Sort by current
                speed_indices.sort(key=lambda i: simulation_results['operating_points'][i][1])
                
                # Extract sequence data
                sequence = {
                    'type': 'current_sequence',
                    'fixed_parameter': ('speed', speed),
                    'currents': [simulation_results['operating_points'][i][1] for i in speed_indices],
                    'torque': [simulation_results['performance_results'][i]['torque'] for i in speed_indices],
                    'efficiency': [simulation_results['performance_results'][i]['efficiency'] for i in speed_indices],
                    'power_factor': [simulation_results['performance_results'][i]['power_factor'] for i in speed_indices],
                    'losses': [simulation_results['thermal_results'][i]['total_losses'] for i in speed_indices],
                    'temperature': [simulation_results['thermal_results'][i]['temperature_rise'] for i in speed_indices]
                }
                sequences.append(sequence)
        
        return sequences
    
    def create_efficiency_contour_sequences(self, simulation_results, efficiency_levels=5):
        """
        Create sequences following efficiency contours
        """
        sequences = []
        
        # Get efficiency data
        efficiencies = [result['efficiency'] for result in simulation_results['performance_results']]
        min_eff, max_eff = min(efficiencies), max(efficiencies)
        
        # Create efficiency levels
        target_efficiencies = np.linspace(min_eff + 0.05, max_eff - 0.05, efficiency_levels)
        
        for target_eff in target_efficiencies:
            # Find points near this efficiency level
            eff_indices = [i for i, eff in enumerate(efficiencies) 
                          if abs(eff - target_eff) < 0.02]
            
            if len(eff_indices) > 3:
                # Sort by speed to create a logical sequence
                eff_indices.sort(key=lambda i: simulation_results['operating_points'][i][0])
                
                sequence = {
                    'type': 'efficiency_contour',
                    'target_efficiency': target_eff,
                    'speeds': [simulation_results['operating_points'][i][0] for i in eff_indices],
                    'currents': [simulation_results['operating_points'][i][1] for i in eff_indices],
                    'efficiencies': [simulation_results['performance_results'][i]['efficiency'] for i in eff_indices],
                    'torque': [simulation_results['performance_results'][i]['torque'] for i in eff_indices],
                    'power': [simulation_results['performance_results'][i]['mechanical_power'] for i in eff_indices]
                }
                sequences.append(sequence)
        
        return sequences
    
    def create_mixed_sequences(self, simulation_results, sequence_length=10, n_sequences=20):
        """
        Create random sequences transitioning between operating points
        """
        sequences = []
        n_points = len(simulation_results['operating_points'])
        
        for _ in range(n_sequences):
            # Random starting point
            start_idx = np.random.randint(0, n_points - sequence_length)
            
            # Create random walk through operating points
            indices = [start_idx]
            current_idx = start_idx
            
            for _ in range(sequence_length - 1):
                # Find neighboring points
                current_speed, current_current = simulation_results['operating_points'][current_idx]
                
                # Calculate distances to all other points
                distances = []
                for i in range(n_points):
                    if i != current_idx:
                        speed, current = simulation_results['operating_points'][i]
                        dist = np.sqrt((speed - current_speed)**2 + (current - current_current)**2)
                        distances.append((dist, i))
                
                # Choose from nearby points with some probability
                distances.sort()
                nearby_indices = [idx for _, idx in distances[:min(10, len(distances))]]
                next_idx = np.random.choice(nearby_indices)
                indices.append(next_idx)
                current_idx = next_idx
            
            # Extract sequence data
            sequence = {
                'type': 'mixed_sequence',
                'speeds': [simulation_results['operating_points'][i][0] for i in indices],
                'currents': [simulation_results['operating_points'][i][1] for i in indices],
                'torque': [simulation_results['performance_results'][i]['torque'] for i in indices],
                'efficiency': [simulation_results['performance_results'][i]['efficiency'] for i in indices],
                'power_factor': [simulation_results['performance_results'][i]['power_factor'] for i in indices],
                'losses': [simulation_results['thermal_results'][i]['total_losses'] for i in indices]
            }
            sequences.append(sequence)
        
        return sequences
    
    def prepare_rnn_data(self, sequences, design_params, sequence_length=10):
        """
        Prepare data for RNN training
        """
        X_sequences = []
        y_sequences = []
        
        for sequence in sequences:
            # Skip sequences that are too short
            if len(sequence['speeds']) < sequence_length:
                continue
            
            # Create input features (speed, current) and targets (performance metrics)
            for i in range(len(sequence['speeds']) - sequence_length + 1):
                # Input sequence: speed and current
                X_seq = []
                for j in range(sequence_length):
                    speed = sequence['speeds'][i + j]
                    current = sequence['currents'][i + j] if 'currents' in sequence else sequence.get('fixed_current', 100)
                    
                    # Normalize inputs
                    speed_norm = speed / 6000  # Normalize to [0, 1]
                    current_norm = current / 200  # Normalize to [0, 1]
                    
                    X_seq.append([speed_norm, current_norm])
                
                # Target sequence: performance metrics
                y_seq = []
                for j in range(sequence_length):
                    torque = sequence['torque'][i + j]
                    efficiency = sequence['efficiency'][i + j]
                    power_factor = sequence['power_factor'][i + j]
                    
                    # Normalize outputs
                    torque_norm = torque / 300  # Normalize to [0, 1]
                    efficiency_norm = efficiency  # Already [0, 1]
                    power_factor_norm = power_factor  # Already [0, 1]
                    
                    y_seq.append([torque_norm, efficiency_norm, power_factor_norm])
                
                X_sequences.append(X_seq)
                y_sequences.append(y_seq)
        
        return np.array(X_sequences), np.array(y_sequences), design_params

# Generate sequence data from simulation results
seq_generator = SequenceDataGenerator()

# Display sequence types
print("Available Sequence Types:")
print("=" * 30)
for seq_type, description in seq_generator.sequence_types.items():
    print(f"{seq_type}: {description}")

### 3.3.1 Demonstration of Sequence Generation

Let's demonstrate the sequence generation process using our simulation results.

In [ ]:
# Create different types of sequences
current_values = [40, 80, 120]  # Fixed current values for speed sequences
speed_values = [2000, 4000, 6000]  # Fixed speed values for current sequences

print("Generating sequences from simulation results...")

# Generate speed sequences
speed_sequences = seq_generator.create_speed_sequences(simulation_results, current_values)
print(f"Generated {len(speed_sequences)} speed sequences")

# Generate current sequences
current_sequences = seq_generator.create_current_sequences(simulation_results, speed_values)
print(f"Generated {len(current_sequences)} current sequences")

# Generate efficiency contour sequences
efficiency_sequences = seq_generator.create_efficiency_contour_sequences(simulation_results)
print(f"Generated {len(efficiency_sequences)} efficiency contour sequences")

# Generate mixed sequences
mixed_sequences = seq_generator.create_mixed_sequences(simulation_results, sequence_length=12, n_sequences=15)
print(f"Generated {len(mixed_sequences)} mixed sequences")

# Combine all sequences
all_sequences = speed_sequences + current_sequences + efficiency_sequences + mixed_sequences
print(f"\nTotal sequences generated: {len(all_sequences)}")

# Prepare RNN training data
X_data, y_data, design_params = seq_generator.prepare_rnn_data(all_sequences, sample_design, sequence_length=8)

print(f"\nRNN Training Data Shape:")
print(f"  Input sequences (X): {X_data.shape}")
print(f"  Target sequences (y): {y_data.shape}")
print(f"  Sequence length: {X_data.shape[1]}")
print(f"  Input features: {X_data.shape[2]} (speed, current)")
print(f"  Output features: {y_data.shape[2]} (torque, efficiency, power_factor)")

### 3.3.2 Visualizing Sequence Data

Let's visualize the different types of sequences to understand their characteristics.

In [ ]:
def plot_sequences(sequences, n_cols=3):
    """
    Visualize different types of sequences
    """
    n_sequences = min(6, len(sequences))  # Show up to 6 sequences
    n_rows = (n_sequences + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = axes
    else:
        axes = axes.flatten()
    
    for i, sequence in enumerate(sequences[:n_sequences]):
        ax = axes[i]
        
        # Create secondary y-axis for efficiency
        ax2 = ax.twinx()
        
        # Plot torque and efficiency
        line1 = ax.plot(sequence['speeds'], sequence['torque'], 
                       'b-o', label='Torque', linewidth=2, markersize=6)
        line2 = ax2.plot(sequence['speeds'], sequence['efficiency'], 
                        'r-s', label='Efficiency', linewidth=2, markersize=6)
        
        ax.set_xlabel('Speed (RPM)', fontsize=10)
        ax.set_ylabel('Torque (Nm)', color='b', fontsize=10)
        ax2.set_ylabel('Efficiency', color='r', fontsize=10)
        
        ax.tick_params(axis='y', labelcolor='b')
        ax2.tick_params(axis='y', labelcolor='r')
        
        # Title with sequence type
        seq_type = sequence['type'].replace('_', ' ').title()
        if 'fixed_parameter' in sequence:
            param_name, param_value = sequence['fixed_parameter']
            title = f'{seq_type}\n({param_name}: {param_value})'
        elif 'target_efficiency' in sequence:
            title = f'{seq_type}\n(Target: {sequence["target_efficiency"]:.2f})'
        else:
            title = seq_type
        
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3)
        
        # Combined legend
        lines = line1 + line2
        labels = [l.get_label() for l in lines]
        ax.legend(lines, labels, loc='best', fontsize=9)
    
    # Hide unused subplots
    for i in range(n_sequences, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.show()

def plot_sequence_statistics(sequences):
    """
    Plot statistics of sequence data
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Collect data from all sequences
    all_speeds = []
    all_torques = []
    all_efficiencies = []
    sequence_lengths = []
    
    for sequence in sequences:
        all_speeds.extend(sequence['speeds'])
        all_torques.extend(sequence['torque'])
        all_efficiencies.extend(sequence['efficiency'])
        sequence_lengths.append(len(sequence['speeds']))
    
    # Speed distribution
    axes[0, 0].hist(all_speeds, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 0].set_title('Speed Distribution Across All Sequences', fontweight='bold')
    axes[0, 0].set_xlabel('Speed (RPM)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Torque distribution
    axes[0, 1].hist(all_torques, bins=20, alpha=0.7, color='lightcoral', edgecolor='black')
    axes[0, 1].set_title('Torque Distribution Across All Sequences', fontweight='bold')
    axes[0, 1].set_xlabel('Torque (Nm)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Efficiency distribution
    axes[1, 0].hist(all_efficiencies, bins=20, alpha=0.7, color='lightgreen', edgecolor='black')
    axes[1, 0].set_title('Efficiency Distribution Across All Sequences', fontweight='bold')
    axes[1, 0].set_xlabel('Efficiency')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Sequence length distribution
    axes[1, 1].hist(sequence_lengths, bins=10, alpha=0.7, color='gold', edgecolor='black')
    axes[1, 1].set_title('Sequence Length Distribution', fontweight='bold')
    axes[1, 1].set_xlabel('Sequence Length')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Visualize sequences
print("Visualizing sample sequences:")
plot_sequences(all_sequences[:6])

# Plot sequence statistics
print("\nSequence data statistics:")
plot_sequence_statistics(all_sequences)

## 3.4 Data Preprocessing and Normalization

Proper data preprocessing is crucial for RNN training. We'll implement normalization techniques and data augmentation strategies.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

class DataPreprocessor:
    """
    Data preprocessing and normalization for RNN training
    """
    
    def __init__(self, scaler_type='minmax'):
        self.scaler_type = scaler_type
        self.scalers = {}
        self.feature_stats = {}
    
    def fit_scalers(self, X_data, y_data):
        """
        Fit normalization scalers
        """
        # Reshape data for fitting (combine batch and sequence dimensions)
        X_reshaped = X_data.reshape(-1, X_data.shape[-1])
        y_reshaped = y_data.reshape(-1, y_data.shape[-1])
        
        # Create scalers
        if self.scaler_type == 'standard':
            self.scalers['X'] = StandardScaler()
            self.scalers['y'] = StandardScaler()
        elif self.scaler_type == 'minmax':
            self.scalers['X'] = MinMaxScaler(feature_range=(-1, 1))
            self.scalers['y'] = MinMaxScaler(feature_range=(-1, 1))
        elif self.scaler_type == 'robust':
            self.scalers['X'] = RobustScaler()
            self.scalers['y'] = RobustScaler()
        
        # Fit scalers
        self.scalers['X'].fit(X_reshaped)
        self.scalers['y'].fit(y_reshaped)
        
        # Calculate feature statistics
        self.feature_stats = {
            'X_mean': np.mean(X_reshaped, axis=0),
            'X_std': np.std(X_reshaped, axis=0),
            'y_mean': np.mean(y_reshaped, axis=0),
            'y_std': np.std(y_reshaped, axis=0),
            'X_min': np.min(X_reshaped, axis=0),
            'X_max': np.max(X_reshaped, axis=0),
            'y_min': np.min(y_reshaped, axis=0),
            'y_max': np.max(y_reshaped, axis=0)
        }
    
    def transform_data(self, X_data, y_data):
        """
        Transform data using fitted scalers
        """
        # Reshape for transformation
        original_X_shape = X_data.shape
        original_y_shape = y_data.shape
        
        X_reshaped = X_data.reshape(-1, X_data.shape[-1])
        y_reshaped = y_data.reshape(-1, y_data.shape[-1])
        
        # Transform
        X_transformed = self.scalers['X'].transform(X_reshaped)
        y_transformed = self.scalers['y'].transform(y_reshaped)
        
        # Reshape back
        X_transformed = X_transformed.reshape(original_X_shape)
        y_transformed = y_transformed.reshape(original_y_shape)
        
        return X_transformed, y_transformed
    
    def inverse_transform_y(self, y_data):
        """
        Inverse transform target data
        """
        original_shape = y_data.shape
        y_reshaped = y_data.reshape(-1, y_data.shape[-1])
        y_original = self.scalers['y'].inverse_transform(y_reshaped)
        return y_original.reshape(original_shape)
    
    def add_noise_augmentation(self, X_data, y_data, noise_factor=0.02):
        """
        Add noise for data augmentation
        """
        # Add Gaussian noise to inputs
        X_noisy = X_data + np.random.normal(0, noise_factor, X_data.shape)
        
        # Add smaller noise to outputs
        y_noisy = y_data + np.random.normal(0, noise_factor * 0.5, y_data.shape)
        
        return X_noisy, y_noisy
    
    def create_temporal_augmentation(self, X_data, y_data, max_shift=2):
        """
        Create temporal augmentations by shifting sequences
        """
        augmented_X = []
        augmented_y = []
        
        for i in range(len(X_data)):
            X_seq = X_data[i]
            y_seq = y_data[i]
            
            # Original sequence
            augmented_X.append(X_seq)
            augmented_y.append(y_seq)
            
            # Shifted sequences
            for shift in range(1, max_shift + 1):
                if len(X_seq) > shift:
                    # Shift forward
                    X_shifted = np.roll(X_seq, shift, axis=0)
                    y_shifted = np.roll(y_seq, shift, axis=0)
                    
                    augmented_X.append(X_shifted)
                    augmented_y.append(y_shifted)
        
        return np.array(augmented_X), np.array(augmented_y)

class MotorDataset(Dataset):
    """
    PyTorch dataset for motor sequence data
    """
    
    def __init__(self, X_data, y_data, design_params=None):
        self.X_data = torch.FloatTensor(X_data)
        self.y_data = torch.FloatTensor(y_data)
        self.design_params = design_params
        
        if self.design_params is not None:
            self.design_params = torch.FloatTensor(list(design_params.values()))
    
    def __len__(self):
        return len(self.X_data)
    
    def __getitem__(self, idx):
        if self.design_params is not None:
            return self.X_data[idx], self.y_data[idx], self.design_params
        else:
            return self.X_data[idx], self.y_data[idx]

# Initialize preprocessor
preprocessor = DataPreprocessor(scaler_type='minmax')

# Fit scalers
preprocessor.fit_scalers(X_data, y_data)

# Transform data
X_normalized, y_normalized = preprocessor.transform_data(X_data, y_data)

print("Data Preprocessing Results:")
print("=" * 30)
print(f"Original data shape: {X_data.shape}, {y_data.shape}")
print(f"Normalized data shape: {X_normalized.shape}, {y_normalized.shape}")
print(f"\nFeature statistics:")
print(f"  X mean: {preprocessor.feature_stats['X_mean']}")
print(f"  X std: {preprocessor.feature_stats['X_std']}")
print(f"  y mean: {preprocessor.feature_stats['y_mean']}")
print(f"  y std: {preprocessor.feature_stats['y_std']}")

# Create data augmentation
X_augmented, y_augmented = preprocessor.add_noise_augmentation(X_normalized, y_normalized)
print(f"\nData augmentation:")
print(f"  Original samples: {len(X_normalized)}")
print(f"  Augmented samples: {len(X_augmented)}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_augmented, y_augmented, test_size=0.2, random_state=42
)

print(f"\nData split:")
print(f"  Training samples: {len(X_train)}")
print(f"  Test samples: {len(X_test)}")

# Create PyTorch datasets
train_dataset = MotorDataset(X_train, y_train, sample_design)
test_dataset = MotorDataset(X_test, y_test, sample_design)

print(f"\nPyTorch datasets created:")
print(f"  Training dataset size: {len(train_dataset)}")
print(f"  Test dataset size: {len(test_dataset)}")

### 3.4.1 Data Quality Validation

Let's validate the quality of our preprocessed data.

In [ ]:
def validate_data_quality(X_data, y_data, preprocessor):
    """
    Validate data quality after preprocessing
    """
    print("Data Quality Validation:")
    print("=" * 25)
    
    # Check for NaN or infinite values
    X_nan_count = np.sum(np.isnan(X_data))
    y_nan_count = np.sum(np.isnan(y_data))
    X_inf_count = np.sum(np.isinf(X_data))
    y_inf_count = np.sum(np.isinf(y_data))
    
    print(f"NaN values - X: {X_nan_count}, y: {y_nan_count}")
    print(f"Infinite values - X: {X_inf_count}, y: {y_inf_count}")
    
    # Check data ranges
    X_min, X_max = np.min(X_data), np.max(X_data)
    y_min, y_max = np.min(y_data), np.max(y_data)
    
    print(f"\nData ranges:")
    print(f"  X: [{X_min:.3f}, {X_max:.3f}]")
    print(f"  y: [{y_min:.3f}, {y_max:.3f}]")
    
    # Check sequence statistics
    seq_lengths = [len(seq) for seq in X_data]
    print(f"\nSequence statistics:")
    print(f"  Min length: {min(seq_lengths)}")
    print(f"  Max length: {max(seq_lengths)}")
    print(f"  Mean length: {np.mean(seq_lengths):.1f}")
    
    # Check data distribution
    X_reshaped = X_data.reshape(-1, X_data.shape[-1])
    y_reshaped = y_data.reshape(-1, y_data.shape[-1])
    
    print(f"\nFeature statistics (normalized):")
    for i in range(X_reshaped.shape[1]):
        feature_name = ['Speed', 'Current'][i]
        mean_val = np.mean(X_reshaped[:, i])
        std_val = np.std(X_reshaped[:, i])
        print(f"  {feature_name}: mean={mean_val:.3f}, std={std_val:.3f}")
    
    for i in range(y_reshaped.shape[1]):
        feature_name = ['Torque', 'Efficiency', 'Power Factor'][i]
        mean_val = np.mean(y_reshaped[:, i])
        std_val = np.std(y_reshaped[:, i])
        print(f"  {feature_name}: mean={mean_val:.3f}, std={std_val:.3f}")

def plot_normalized_data(X_data, y_data):
    """
    Visualize normalized data distribution
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Reshape data
    X_reshaped = X_data.reshape(-1, X_data.shape[-1])
    y_reshaped = y_data.reshape(-1, y_data.shape[-1])
    
    feature_names = ['Speed', 'Current', 'Torque', 'Efficiency', 'Power Factor']
    all_features = np.concatenate([X_reshaped, y_reshaped], axis=1)
    
    for i, feature_name in enumerate(feature_names):
        row = i // 3
        col = i % 3
        ax = axes[row, col]
        
        # Plot histogram
        ax.hist(all_features[:, i], bins=30, alpha=0.7, edgecolor='black')
        ax.set_title(f'{feature_name} Distribution (Normalized)', fontweight='bold')
        ax.set_xlabel('Normalized Value')
        ax.set_ylabel('Frequency')
        ax.grid(True, alpha=0.3)
        
        # Add statistics text
        mean_val = np.mean(all_features[:, i])
        std_val = np.std(all_features[:, i])
        ax.text(0.05, 0.95, f'μ={mean_val:.3f}\nσ={std_val:.3f}', 
               transform=ax.transAxes, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()

# Validate data quality
validate_data_quality(X_train, y_train, preprocessor)

# Plot normalized data distribution
plot_normalized_data(X_train, y_train)

## 3.5 Summary and Key Takeaways

### 3.5.1 What We Covered

In this chapter, we covered the complete data generation and processing pipeline for RNN-based performance map prediction:

**1. Latin Hypercube Sampling:**
- Efficient design space exploration with optimal space-filling properties
- Comparison with traditional sampling methods (random, grid)
- Implementation for 20 motor design parameters
- Quality metrics and coverage analysis

**2. 4-Stage FE Simulation Process:**
- Magnetic analysis: Field distribution and flux linkage
- Thermal analysis: Temperature distribution and losses
- Mechanical analysis: Stress and vibration characteristics
- Performance analysis: Efficiency, torque, and power metrics
- Computational cost analysis and optimization strategies

**3. Sequence Data Generation:**
- Speed sequences: Fixed current, varying speed
- Current sequences: Fixed speed, varying current
- Efficiency contour sequences: Following constant efficiency lines
- Mixed sequences: Random transitions between operating points
- Data preparation for RNN training with proper normalization

**4. Data Preprocessing:**
- Multiple normalization strategies (Standard, MinMax, Robust)
- Data augmentation through noise injection
- Temporal augmentation through sequence shifting
- Quality validation and statistical analysis

### 3.5.2 Key Implementation Details

**Latin Hypercube Sampling Advantages:**
- Provides excellent coverage with fewer samples
- Reduces correlation between parameters
- Enables efficient exploration of high-dimensional design spaces
- Suitable for expensive FE simulations

**4-Stage FE Process Benefits:**
- Comprehensive analysis of motor behavior
- Modular approach allows selective stage execution
- Different computational costs enable optimization
- Rich feature set for RNN training

**Sequence Representation Strategy:**
- Captures temporal relationships between operating points
- Multiple sequence types for comprehensive learning
- Proper normalization for stable RNN training
- Augmentation techniques to improve generalization

### 3.5.3 Computational Considerations

**Time Complexity:**
- LHS generation: O(n_samples × n_parameters)
- 4-stage FE simulation: O(n_designs × n_op_points × stage_time)
- Sequence generation: O(n_simulation_points)
- Data preprocessing: O(n_sequences × sequence_length)

**Memory Requirements:**
- Design samples: O(n_designs × n_parameters)
- Simulation results: O(n_designs × n_op_points × n_features)
- Sequence data: O(n_sequences × seq_length × n_features)
- Augmented data: 2-3× original sequence data size

**Optimization Strategies:**
- Parallel processing for independent simulations
- Adaptive sampling for design space exploration
- Selective stage execution based on requirements
- Efficient data structures for sequence handling

### 3.5.4 Best Practices

**1. Design Space Sampling:**
- Use LHS for high-dimensional spaces
- Validate sample quality with correlation analysis
- Consider parameter constraints and physical limits
- Include edge cases and boundary conditions

**2. FE Simulation:**
- Validate each stage independently
- Use appropriate mesh densities for accuracy
- Consider simplified models for rapid prototyping
- Implement convergence criteria and quality checks

**3. Sequence Generation:**
- Create diverse sequence types for comprehensive learning
- Ensure sequence lengths are appropriate for RNN architecture
- Balance sequence diversity with computational cost
- Validate physical consistency of sequences

**4. Data Preprocessing:**
- Choose appropriate normalization for the data distribution
- Validate data quality after preprocessing
- Use data augmentation to improve generalization
- Maintain original data ranges for interpretation

### 3.5.5 Next Steps

With our comprehensive dataset prepared, we're ready to move to the next chapter where we'll explore:

- **RNN Architecture Fundamentals:** GRU cells, bidirectional processing
- **Attention Mechanisms:** Encoder-decoder architectures, context vectors
- **Model Implementation:** Modular vs end-to-end approaches
- **Training Strategies:** Loss functions, optimization, regularization

The data generation pipeline we've established provides a solid foundation for training accurate and robust RNN models for motor performance map prediction.